# Candidate Selector

csv file structure:
```
entity_label,start_position,end_position,correct_prediction,candidates,matching_candidate_index
Bandar Seri Begawan,0,19,True,"[[""Bandar Seri Begawan"", 0.942, 1.0, 0.839, 0.354], [""Pusat Bandar, Brunei"", 0.937, 0.46, 0.289, 0.348], [""1999 Southeast Asian Games"", 0.943, 0.27, 0.349, 0.189], [""List of diplomatic missions of Russia"", 0, 0.21, 1.0, 0.087], [""List of diplomatic missions of India"", 0, 0.22, 0.889, 0.161], [""Embassy of the Philippines, Bandar Seri Begawan"", 0, 0.58, 0.124, 0.45], [""Bandar Seri Begawan"", 0, 1.0, 0.0, 0.147], [""Bandar Seri Begawan"", 0, 1.0, 0.0, 0.147], [""Flughafen Bandar Seri Begawan"", 0, 0.79, 0.0, 0.147], [""Deutsche Botschaft Bandar Seri Begawan"", 0, 0.67, 0.0, 0.147]]",0
AFP,26,29,False,"[[""AFC Ajax"", 0.694, 0.36, 0.937, -0.12], [""AFC Wimbledon"", 0.694, 0.25, 0.806, 0.007], [""Sunderland A.F.C."", 0.694, 0.2, 1.0, -0.148], [""Armed Forces of the Philippines"", 0.692, 0.18, 0.463, 0.302], [""Agence France-Presse"", 0.652, 0.26, 0.356, 0.226], [""Alpha-fetoprotein"", 0.887, 0.1, 0.345, 0.057], [""AFP"", 0, 1.0, 0.056, 0.147], [""AFP-L3"", 0, 0.67, 0.0, 0.147], [""Armed Forces of the Philippines Medal of Valor"", 0, 0.12, 0.423, 0.126], [""AFP Visayas Command"", 0, 0.27, 0.0, 0.147]]",4
White House,343,354,False,"[[""White"", 0.874, 0.62, 1.0, -0.015], [""White House"", 0.76, 1.0, 0.295, 0.072], [""Chicago White Sox"", 0.686, 0.5, 0.888, -0.056], [""Vale of White Horse"", 0.813, 0.67, 0.471, -0.09], [""White House Office"", 0.707, 0.76, 0.0, 0.133], [""White-shouldered house moth"", 0.844, 0.53, 0.096, 0.085], [""White House Correspondents' Association"", 0.691, 0.44, 0.115, 0.223], [""White House, Tennessee"", 0.809, 0.67, 0.041, -0.061], [""White House Press Secretary"", 0, 0.58, 0.028, 0.216], [""White House COVID-19 outbreak"", 0, 0.55, 0.123, 0.144]]",1

```

In [1]:
import pandas as pd
import ast

csv_filepath = "/content/logs.csv"
# Load the CSV file
df = pd.read_csv(csv_filepath)

# Convert the 'candidates' column from string to list of lists
df["candidates"] = df["candidates"].apply(ast.literal_eval)

# Display first few rows
print(df.head())


          entity_label  start_position  end_position  correct_prediction  \
0  Bandar Seri Begawan               0            19                True   
1                  AFP              26            29               False   
2          White House             343           354               False   
3             NPR NEWS              16            24                True   
4           Washington              28            38                True   

                                          candidates  matching_candidate_index  
0  [[Bandar Seri Begawan, 0.942, 1.0, 0.839, 0.35...                         0  
1  [[AFC Ajax, 0.694, 0.36, 0.937, -0.12], [AFC W...                         4  
2  [[White, 0.874, 0.62, 1.0, -0.015], [White Hou...                         1  
3  [[NPR, 0.747, 0.55, 0.629, 0.067], [American B...                         0  
4  [[Washington, D.C., 0.848, 0.77, 1.0, -0.011],...                         0  


In [2]:
import numpy as np

def preprocess_data(df):
    X, y = [], []

    for _, row in df.iterrows():
        candidates = row["candidates"]
        matching_index = row["matching_candidate_index"]

        # Extract only the 4 float metrics from each candidate
        features = [metrics[1:] for metrics in candidates]  # Ignore entity name

        X.append(np.array(features).flatten())  # Flatten into a single vector
        y.append(matching_index)

    return np.array(X), np.array(y)

X, y = preprocess_data(df)

print("Feature shape:", X.shape)  # Expect (num_samples, 40)
print("Target shape:", y.shape)   # Expect (num_samples,)


Feature shape: (181, 40)
Target shape: (181,)


## RandomForestClassifier

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Train a Random Forest model
model = RandomForestClassifier(n_estimators=1000)
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")


Model Accuracy: 0.8182


In [16]:
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier(n_estimators=1000, random_state=37)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print("Mean Accuracy:", cv_scores.mean())
print("Standard Deviation:", cv_scores.std())


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Mean Accuracy: 0.8232732732732734
Standard Deviation: 0.02152233627435587


## Neural network

In [86]:
import pandas as pd
import ast
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# Load CSV
df = pd.read_csv('./logs.csv')

# Parse candidates column from string to list of lists
df['candidates'] = df['candidates'].apply(ast.literal_eval)

# Extract input features (X) and labels (y)
X = torch.tensor([entry[1:] for candidates in df['candidates'] for entry in candidates], dtype=torch.float32)
y = torch.tensor(df['matching_candidate_index'].values, dtype=torch.long)

# Reshape X into (num_samples, 10, 4) where 10 candidates each have 4 metrics
X = X.view(len(df), 10, 4)  # Each row corresponds to 10 candidates


In [87]:
class CandidateSelectorNN(nn.Module):
    def __init__(self):
        super(CandidateSelectorNN, self).__init__()
        self.fc1 = nn.Linear(4, 16)  # Input: 4 metrics -> Hidden layer
        self.fc2 = nn.Linear(16, 8)  # Hidden layer
        self.fc3 = nn.Linear(8, 1)   # Output: Score for each candidate

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze(-1)  # Shape: (batch_size, 10)


In [88]:
class CandidateDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CandidateDataset(X, y)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CandidateSelectorNN().to(device)
criterion = nn.CrossEntropyLoss()  # Suitable for classification tasks
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)  # Shape: (batch_size, 10)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {correct / total:.4f}")


Epoch 1, Loss: 20.4690, Accuracy: 0.5556
Epoch 2, Loss: 20.3743, Accuracy: 0.7014
Epoch 3, Loss: 20.2668, Accuracy: 0.7847
Epoch 4, Loss: 20.1386, Accuracy: 0.7986
Epoch 5, Loss: 19.9840, Accuracy: 0.8194
Epoch 6, Loss: 19.8038, Accuracy: 0.8264
Epoch 7, Loss: 19.5954, Accuracy: 0.8333
Epoch 8, Loss: 19.3574, Accuracy: 0.8264
Epoch 9, Loss: 19.0873, Accuracy: 0.8264
Epoch 10, Loss: 18.7843, Accuracy: 0.8264
Epoch 11, Loss: 18.4436, Accuracy: 0.8264
Epoch 12, Loss: 18.0730, Accuracy: 0.8333
Epoch 13, Loss: 17.6648, Accuracy: 0.8333
Epoch 14, Loss: 17.2141, Accuracy: 0.8333
Epoch 15, Loss: 16.7289, Accuracy: 0.8264
Epoch 16, Loss: 16.1919, Accuracy: 0.8264
Epoch 17, Loss: 15.6205, Accuracy: 0.8264
Epoch 18, Loss: 15.0141, Accuracy: 0.8264
Epoch 19, Loss: 14.3883, Accuracy: 0.8264
Epoch 20, Loss: 13.7440, Accuracy: 0.8264
Epoch 21, Loss: 13.0802, Accuracy: 0.8264
Epoch 22, Loss: 12.3932, Accuracy: 0.8194
Epoch 23, Loss: 11.6568, Accuracy: 0.8194
Epoch 24, Loss: 10.8121, Accuracy: 0.8264
E

In [90]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test Accuracy: {correct / total:.4f}")


Test Accuracy: 0.8649


In [91]:
torch.save(model.state_dict(), "candidate_selector_model.pth")


In [85]:
# Recreate the model architecture
model = CandidateSelectorNN()

# Load saved weights
model.load_state_dict(torch.load("candidate_selector_model.pth"))

# Set model to evaluation mode
model.eval()


CandidateSelectorNN(
  (fc1): Linear(in_features=4, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
)

## LSTM


In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split

# Example network
class CandidateSelectorRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(CandidateSelectorRNN, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)  # Output 10 classes (candidates)

    def forward(self, x):
        # x shape: (batch_size, num_candidates, num_features)
        lstm_out, (hn, cn) = self.lstm(x)
        final_hidden_state = hn[-1]  # Shape: (batch_size, hidden_dim)
        out = self.fc(final_hidden_state)  # Shape: (batch_size, 10)
        return out


In [46]:

# Create Dataset (Assume you already have X and y)
class CandidateDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)


In [47]:

# Parameters
input_dim = 4  # Number of features per candidate (e.g., 4 metrics)
hidden_dim = 64
output_dim = 10  # Number of candidates
epochs = 20
batch_size = 16
learning_rate = 0.001

# Dataset preparation (make sure X and y are properly set)
dataset = CandidateDataset(X, y)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize the model
model = CandidateSelectorRNN(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Expects target as integer indices
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)  # Shape: (batch_size, 10)

        # Ensure target is in the range [0, 9] for 10 candidates
        loss = criterion(outputs, y_batch)  # y_batch should be integer labels (0 to 9)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {correct / total:.4f}")



<ipython-input-46-85f6a0109ef3>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)


Epoch 1, Loss: 20.8257, Accuracy: 0.0347
Epoch 2, Loss: 18.9207, Accuracy: 0.7361
Epoch 3, Loss: 13.9673, Accuracy: 0.7569
Epoch 4, Loss: 9.2824, Accuracy: 0.7569
Epoch 5, Loss: 9.3076, Accuracy: 0.7569
Epoch 6, Loss: 8.8379, Accuracy: 0.7569
Epoch 7, Loss: 8.7409, Accuracy: 0.7569
Epoch 8, Loss: 8.6878, Accuracy: 0.7569
Epoch 9, Loss: 8.6492, Accuracy: 0.7569
Epoch 10, Loss: 8.6549, Accuracy: 0.7569
Epoch 11, Loss: 8.6148, Accuracy: 0.7569
Epoch 12, Loss: 8.5896, Accuracy: 0.7569
Epoch 13, Loss: 8.5723, Accuracy: 0.7569
Epoch 14, Loss: 8.5653, Accuracy: 0.7569
Epoch 15, Loss: 8.5470, Accuracy: 0.7569
Epoch 16, Loss: 8.5375, Accuracy: 0.7569
Epoch 17, Loss: 8.5380, Accuracy: 0.7569
Epoch 18, Loss: 8.5203, Accuracy: 0.7569
Epoch 19, Loss: 8.4886, Accuracy: 0.7569
Epoch 20, Loss: 8.4822, Accuracy: 0.7569


In [48]:

# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += y_batch.size(0)

print(f"Test Accuracy: {correct / total:.4f}")

Test Accuracy: 0.5676


<ipython-input-46-85f6a0109ef3>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)
